In [ ]:
from huggingface_hub import notebook_login 
notebook_login()

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -qq -U evaluate rouge_score

In [2]:
import unsloth
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-17 07:15:05.282592: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773731705.436300      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773731705.485927      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773731705.866881      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773731705.866922      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773731705.866925      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1" 
max_seq_length = 2048 

In [22]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,   # Define context length
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # Add your token if using a gated model
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-0.6B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

R_model, R_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Rohit-Katkar2003/qwen-0.6b-fine-tune",  # your HF repo
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-0.6B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [24]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [7]:
# Add LoRA adapters to the model
R_model = FastLanguageModel.get_peft_model(
    R_model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — LOAD + COMBINE DATASETS
# ─────────────────────────────────────────────────────────────────────────────
print("[ 2/5 ] Loading datasets") 
from datasets import load_dataset  , concatenate_datasets
import random 
import gc 
SEED = 42 

SEED           = 42
TARGET_TOTAL   = 20000        # 20k is sufficient — quality >> quantity for SLMs

WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
}

ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train")

print(f"    D1={len(ds1):,}  D2={len(ds2):,}  D3={len(ds3):,}  D4={len(ds4):,}")

random.seed(SEED)
splits = []
for name, ds, w in [
    ("instruction",   ds1, WEIGHTS["instruction"]),
    ("summarization", ds2, WEIGHTS["summarization"]),
    ("extraction",    ds3, WEIGHTS["extraction"]),
    ("decomposition", ds4, WEIGHTS["decomposition"]),
]:
    target_n = int(TARGET_TOTAL * w)
    if len(ds) >= target_n:
        idx = random.sample(range(len(ds)), target_n)
    else:
        repeats = (target_n // len(ds)) + 1
        idx     = (list(range(len(ds))) * repeats)[:target_n]
        random.shuffle(idx)
        print(f"    Upsampled {name}: {len(ds)} → {target_n}")
    splits.append(ds.select(idx))

combined = concatenate_datasets(splits).shuffle(seed=SEED)
del ds1, ds2, ds3, ds4
gc.collect()
print(f"[ 2/5 ] Combined: {len(combined):,} samples ✓")

[ 2/5 ] Loading datasets
    D1=71,179  D2=100,000  D3=100,000  D4=500
    Upsampled decomposition: 500 → 3000
[ 2/5 ] Combined: 20,000 samples ✓


In [26]:
combined[1]

{'dataset': 'squad_v2',
 'task': 'information_extraction',
 'system': 'You are an information extraction agent. Extract precise facts from text. Return valid JSON. If the answer cannot be found, return null for that field.',
 'messages': [{'role': 'user',
   'content': "Analyze this text and extract the requested fact. If not present, state null.\n\nText:\nIn the 1970s, job losses due to industrial restructuring caused New York City to suffer from economic problems and rising crime rates. While a resurgence in the financial industry greatly improved the city's economic health in the 1980s, New York's crime rate continued to increase through that decade and into the beginning of the 1990s. By the mid 1990s, crime rates started to drop dramatically due to revised police strategies, improving economic opportunities, gentrification, and new residents, both American transplants and new immigrants from Asia and Latin America. Important new sectors, such as Silicon Alley, emerged in the city'

In [27]:
def format_chat(example):
    system = example["system"]
    messages = example["messages"]

    text = f"<|system|>\n{system}\n"

    for msg in messages:
        if msg["role"] == "user":
            text += f"<|user|>\n{msg['content']}\n"
        elif msg["role"] == "assistant":
            text += f"<|assistant|>\n{msg['content']}\n"

    return {"text": text}

train_dataset = combined.map(format_chat)

In [28]:
split_dataset = train_dataset.train_test_split(test_size=0.15, seed=3407) 
# Access the train and validation sets
train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"] 

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))
print(train_dataset[0])

Train size: 17000
Valid size: 3000
{'dataset': 'no_robot', 'task': 'instruction_tune', 'system': 'You are Helpful assistant.', 'messages': [{'role': 'user', 'content': 'What is the biggest continent?'}, {'role': 'assistant', 'content': "There are various ways to measure and compare continents. Asia is the biggest continent on Earth in terms of land mass. Asia's area is 17.2 million square miles. Asia also has the largest population among the seven continents, with over 4.5 billion people. The next largest continent in terms of land mass is Africa. Africa is 11.6 million square miles in area, and is home to 1.3 billion people."}], 'text': "<|system|>\nYou are Helpful assistant.\n<|user|>\nWhat is the biggest continent?\n<|assistant|>\nThere are various ways to measure and compare continents. Asia is the biggest continent on Earth in terms of land mass. Asia's area is 17.2 million square miles. Asia also has the largest population among the seven continents, with over 4.5 billion people.

In [29]:
print("<|assistant|>" in train_dataset[0]["text"])

True


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length
    ) 
    
train_dataset = train_dataset.map(tokenize_function, batched=True, num_proc=1, remove_columns=["text"])
valid_dataset = valid_dataset.map(tokenize_function, batched=True, num_proc=1, remove_columns=["text"]) 

In [30]:
import numpy as np
from transformers import EvalPrediction
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

In [31]:
def preprocess_logits_for_metrics(logits, labels):
    """Returns predicted token IDs (argmax) for metrics calculation"""
    if isinstance(logits, tuple):
        logits = logits[0]  # Unpack if needed
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds
    
    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()
    
    accuracy = (preds_flat == labels_flat).mean()
    
    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    
    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    } 

def compute_metricsR(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds
    
    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()
    
    accuracy = (preds_flat == labels_flat).mean()
    
    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    
    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

In [32]:
from trl import SFTTrainer, SFTConfig
sftconfig = SFTConfig(
    per_device_train_batch_size=2,        # ⬇ reduced from 8
    gradient_accumulation_steps=32,       # keeps effective batch = 64
    warmup_steps=5,
    num_train_epochs=4,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    per_device_eval_batch_size=1,         # ⬇ reduced from 8 — key fix
    seed=3407,
    dataloader_pin_memory=False,          # ⬇ disable to save memory
    output_dir="outputs",
    report_to="none",
    eval_strategy="steps",
    eval_steps=100,                       # ⬇ evaluate less frequently
    fp16_full_eval=False,                 # ⬇ disable — saves memory
    eval_accumulation_steps=4,            # ⬆ accumulate instead of batch
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataset_num_proc=1,
)

In [33]:
# Remove compute_metrics and preprocess_logits_for_metrics entirely
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=sftconfig,
    # No compute_metrics — eval_loss alone is sufficient and memory-safe
)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|user|>\n",
    response_part    = "<|assistant|>\n",
)

Unsloth: Removed 12 out of 17000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.
Unsloth: Removed 1 out of 3000 samples from eval_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [34]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()
print("Training finished.")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,988 | Num Epochs = 4 | Total steps = 1,064
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 32 x 1) = 64
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.397800,1.201740
200,1.124500,1.175696
300,1.232000,1.161830
400,1.190800,1.157244
500,1.041100,1.148186
600,1.011300,1.154884
700,0.957300,1.153537
800,1.060600,1.149496
900,0.978900,1.161019
1000,1.347500,1.159769


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training finished.


In [35]:
# print training stats
print(trainer_stats)

TrainOutput(global_step=1064, training_loss=1.2006952129584505, metrics={'train_runtime': 18398.2864, 'train_samples_per_second': 3.693, 'train_steps_per_second': 0.058, 'total_flos': 6.227161645056e+16, 'train_loss': 1.2006952129584505, 'epoch': 4.0})


In [36]:
import shutil

# Zip the outputs folder
shutil.make_archive("outputs_backup", "zip", "outputs")

print("✅ outputs folder zipped")

✅ outputs folder zipped


In [ ]:
from huggingface_hub import login 
TOKEN = "" 
login(TOKEN) 

model.push_to_hub("Rohit-Katkar2003/qwen-0.6b-fine-tune-4")
tokenizer.push_to_hub("Rohit-Katkar2003/qwen-0.6b-fine-tune-4")

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Rohit-Katkar2003/qwen-0.6b-fine-tune-4


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Rohit-Katkar2003/qwen-0.6b-fine-tune-4"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_response(prompt, max_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

In [42]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Rohit-Katkar2003/qwen-0.6b-fine-tune-4",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
    
print(generate_response("hi"))

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-0.6B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
hi. I'm trying to find the value of the integral of 1/(x^2 + 1) dx from 0 to infinity. I know that the integral of 1/(x^2 + 1) dx from 0 to infinity is pi/2. But I'm not sure why. I


In [ ]:
from IPython.display import FileLink

FileLink("outputs_backup.zip")

In [ ]:
import torch
import gc

def clean_cuda():
    print("Cleaning CUDA memory...")

    # Run Python garbage collection
    gc.collect()

    # Empty PyTorch CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print("CUDA memory cleaned.") 
    
clean_cuda()